In [4]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time 

In [7]:
try:
    #indicamos la url a la que realizaremos peticiones GET
    book_scrape = "https://books.toscrape.com/"
    #donde guardaremos la respuesta que obtenemos de la pagina WEB
    response = requests.get(book_scrape)
    #creamos un objeto soup para parsearlo con el contenido html 
    soup = BeautifulSoup(response.text, "html.parser")
    #verificar estado de conexion 200 = exitosa
    if response.status_code == 200:
        print("Conexion exitosa...")   
except (Exception,KeyboardInterrupt) as e:
    print(f"Hubo un error {e}")


Conexion exitosa...


In [ ]:
libro_datos = [] #creamos una lista que almacenara los datos de los libros
for page_num in range(1,51):

    books_pages = f'https://books.toscrape.com/catalogue/page-{page_num}.html' #realizara las iteraciones del for dentro de la url de cada pagina del catalogo
    response = requests.get(books_pages)
    soup = BeautifulSoup(response.content, 'html.parser') # obtener el html 

    libros = soup.find_all('h3')

    #iterar sobre la lista de titulos de libros
    for libro in libros:
        try:
            libro_url = libro.find('a')['href'] #encontrar la url vinculada al libro (libro.find)
            libro_response = requests.get('https://books.toscrape.com/catalogue/' + libro_url) #accede a la url de books to scrape + a la url de el libro 
            libro_soup = BeautifulSoup(libro_response.content, "html.parser")  #obtenemos el html de la pagina de la url del libro 
            
            #busqueda de datos de los libros mediante etiquetas HTML
            titulo = libro_soup.find('h1').text #encontrar el titulo del libro mediante la etiqueta del h1
            categoria = libro_soup.find('ul', class_ ="breadcrumb").find_all('a')[2].text.strip() # accede a la etiqueta ul donde encuentra la ruta de busqueda donde se encuentra la categoria del libro 
            calificacion = libro_soup.find('p', class_ ='star-rating')['class'][1]
            precio = libro_soup.find('p', class_ ="price_color").text.strip()

            libro_datos.append([titulo,categoria,calificacion,precio])
        except (Exception,KeyboardInterrupt) as e:
            print(f"Hubo un error {e}")
            continue

In [65]:
#convertir una lista en un dataframe
df = pd.DataFrame(libro_datos, columns=["titulo","categoria","calificacion","precio"])
df 

,titulo,categoria,calificacion,precio
0,A Light in the Attic,Poetry,Three,£51.77
1,Tipping the Velvet,Historical Fiction,One,£53.74
2,Soumission,Fiction,One,£50.10
3,Sharp Objects,Mystery,Four,£47.82
4,Sapiens: A Brief History of Humankind,History,Five,£54.23
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,Classics,One,£55.53
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Sequential Art,Four,£57.06
997,A Spy's Devotion (The Regency Spies of London #1),Historical Fiction,Five,£16.97
998,1st to Die (Women's Murder Club #1),Mystery,One,£53.98


In [67]:
# convertir dataframe a csv
df.to_csv("libros_scrapeados.csv", index=False)

In [66]:
df["precio"] = df["precio"].str.replace("£", "").astype(float)

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["calificacion"] = df["calificacion"].map(rating_map)

In [5]:
columna_titulo= ['titulo']
#leemos el csv para poder realizar acciones sobre los campos del csv
datos = pd.read_csv("../data/libros_scrapeados.csv")
titulos = datos['titulo'].tolist()
keys = []
for titulo in titulos:

    try:
        titulo_corto = titulo.split(":")[0].split("(")[0].split(",")[0].strip()
        params = {"title": titulo_corto, "limit": 1}
        url_opl = "https://openlibrary.org/search.json"
        response = requests.get(url_opl, params=params)
        data = response.json()

        doc= data["docs"][0]

        opl_id = doc["author_key"][0]

        keys.append([opl_id])
    except Exception as e:
        print(f"No encontrado: {titulo} - {e}")
        keys.append([None])
        continue

    print (keys)

[['OL548174A']]
[['OL548174A'], ['OL39232A']]
[['OL548174A'], ['OL39232A'], ['OL300477A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A'], ['OL15975018A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A'], ['OL15975018A'], ['OL1353999A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A'], ['OL15975018A'], ['OL1353999A'], ['OL1391804A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A'], ['OL15975018A'], ['OL1353999A'], [

KeyboardInterrupt: 

In [5]:
#convertir una lista en un dataframe
df = pd.DataFrame(keys, columns=["autor_key"])
df 

,autor_key
0,OL548174A
1,OL39232A
2,OL300477A
3,OL1433006A
4,OL3778242A
...,...
995,OL22098A
996,OL7423510A
997,OL7032059A
998,OL22258A


In [8]:
# convertir dataframe a csv
df.to_csv("autor_key.csv", index=False)

In [ ]:
columna_autor_key= ['autor_key']
#leemos el csv para poder realizar acciones sobre los campos del csv
dato_key = pd.read_csv("../data/autor_key.csv")
autor_keys = dato_key['autor_key'].tolist()

autor_datos = []
for key in autor_keys:
    if pd.isna(key):
        continue     
    url_autor = f"https://openlibrary.org/authors/{key}.json"
    response_autor = requests.get(url_autor)
    data_autor = response_autor.json()

    anho_nacimiento = data_autor.get("birth_date", None)
    nombre = data_autor.get("personal_name",None)
    fecha_creacion = data_autor.get("created",None)
    time.sleep(0.5)  # al final de cada iteración del loop
    
    autor_datos.append([anho_nacimiento,nombre,fecha_creacion])

    print(f"Nombre: {nombre}")
    print(f"Anho: {anho_nacimiento}")
    print(f"Creacion: {fecha_creacion}")
    print("--------------")